<a href="https://colab.research.google.com/github/ebmobodo/sentiment-analysis-distilbert/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning DistilBERT for Sentiment Analysis
Author:  Mobodo Ebubechukwu

Goal: In this project, I am going to fine-tune a pre-trained machine learning model called DistilBERT to classify movie reviews as either Positive or Negative.

### 1. Installing Required Libraries
First, I need to install the Hugging Face `transformers` and `datasets` libraries to get my model and data.

In [2]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


 2. Loading the IMDB Dataset
I am using the IMDB movie review dataset. To save time and compute power, I am shrinking the dataset to 1,000 training examples and 500 testing examples.

In [3]:
from datasets import load_dataset

# Added the 'stanfordnlp/' namespace to fix the HfUriError
dataset = load_dataset("stanfordnlp/imdb")

# Shrink the dataset for faster training
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Data loaded! Here is a sample review:")
print(small_train[0]['text'])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Data loaded! Here is a sample review:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...


 3. Tokenizing the Data
AI models can't read English; they only understand numbers. I am using a Tokenizer to convert the movie reviews into a format the DistilBERT model can understand.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)
print("Tokenization complete!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete!


4. Loading the Pre-Trained Model
I am loading `distilbert-base-uncased`. I am telling it to expect 2 labels: 0 for Negative, and 1 for Positive.

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 5. Setting up Metrics
To know if my model is learning, I need to track its accuracy during training.

In [6]:
!pip install evaluate -q
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


 6. Fine-Tuning the Model
Now, I will set up the Training Arguments and start the training process. The model will look at the data 3 times (3 epochs).

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification

# Re-defining model to ensure it's available in this cell's scope
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    # THIS IS THE CHANGED LINE:
    output_dir="/content/drive/MyDrive/my_movie_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.337674,0.872000
2,No log,0.402757,0.876000
3,No log,0.437020,0.876000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.2667992960611979, metrics={'train_runtime': 185.747, 'train_samples_per_second': 16.151, 'train_steps_per_second': 2.019, 'total_flos': 397402195968000.0, 'train_loss': 0.2667992960611979, 'epoch': 3.0})

 7. Testing the Fine-Tuned Model
Training is done! Now I will write my own custom movie reviews and see if the AI can correctly guess if they are positive or negative.
*(Label 1 = Positive, Label 0 = Negative)*

In [13]:
from transformers import pipeline

# Set up a pipeline to use our newly trained model
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

# Write your own reviews here!
my_reviews = [
    "This was the most amazing and breathtaking movie I have ever seen!",
    "I hated this movie. The acting was terrible and the plot made no sense.",
    "The trailer looked amazing and the actors are incredibly talented, but the actual movie was a complete disaster.",
    "The first half was incredibly boring and I almost walked out. But the twist at the end made it the best movie I've seen all year!"
]

results = classifier(my_reviews)

for review, result in zip(my_reviews, results):
    print(f"Review: {review}")
    print(f"Prediction: {result['label']} (Confidence: {result['score']:.2f})\n")

Review: This was the most amazing and breathtaking movie I have ever seen!
Prediction: LABEL_1 (Confidence: 0.99)

Review: I hated this movie. The acting was terrible and the plot made no sense.
Prediction: LABEL_0 (Confidence: 0.99)

Review: The trailer looked amazing and the actors are incredibly talented, but the actual movie was a complete disaster.
Prediction: LABEL_0 (Confidence: 0.97)

Review: The first half was incredibly boring and I almost walked out. But the twist at the end made it the best movie I've seen all year!
Prediction: LABEL_0 (Confidence: 0.60)

